## ChromaDB with Explicit Embeddings
**Goal**

- Create a Chroma DB (in-memory) and create/query/fetch/update/delete a collection

**Steps**
- Create explicit documents and embeddings
- Create a collection from these documents/embeddings
- Query/fetch/update/delete operations
  
**Tech stack**
- Chroma DB
- SentenceTransformers (`all-MiniLM-L6-v2`)

### Install Dependencies

In [2]:
%pip install chromadb sentence-transformers -q

Note: you may need to restart the kernel to use updated packages.


### Setup — Client & Embedding Model

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.Client()

# Local embedding model (384 dimensions)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Client and model ready")

/home/manoj/Venvs/venv-aiml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 238.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Client and model ready


### Create a collection

In [2]:
collection = client.create_collection(
    name="my_collection",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection created: '{collection.name}'")

Collection created: 'my_collection'


### Create documents and embeddings

In [10]:
documents = [
    "Python is a great programming language",
    "Machine learning uses algorithms to learn from data",
    "ChromaDB stores and queries vector embeddings",
    "The sky is blue and the grass is green",
]

embeddings = model.encode(documents).tolist()
print(len(embeddings))
print(f"Embedding dimensions : {len(embeddings[0])}")
print(f"doc1 (first 5 dims)  : {embeddings[0][:5]}")
print(f"doc2 (first 5 dims)  : {embeddings[1][:5]}")
print(f"doc3 (first 5 dims)  : {embeddings[2][:5]}")
print(f"doc4 (first 5 dims)  : {embeddings[3][:5]}")

4
Embedding dimensions : 384
doc1 (first 5 dims)  : [-0.049650564789772034, -0.003535160794854164, -0.005157203879207373, -0.0010899649932980537, -0.05018303543329239]
doc2 (first 5 dims)  : [-0.05912225693464279, 0.01819334737956524, 0.025811830535531044, 0.021062245592474937, 0.08048992604017258]
doc3 (first 5 dims)  : [-0.060502175241708755, -0.03483855351805687, -0.06409699469804764, 0.03493492305278778, -0.0073504336178302765]
doc4 (first 5 dims)  : [0.009375374764204025, 0.04830996319651604, 0.051919229328632355, -0.021829677745699883, 0.056384794414043427]


### Add documents with explicit embeddings

In [11]:
collection.add(
    documents=documents,
    ids=["doc1", "doc2", "doc3", "doc4"],
    embeddings=embeddings,
)

print(f"Stored {collection.count()} documents")

Stored 4 documents


### Query with explicit embedding

In [7]:
query = "What is machine learning?"
query_embedding = model.encode([query]).tolist()
print(f"Query: '{query}'")
print("query_embedding: ", query_embedding[0][:5])

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    include=["documents", "embeddings", "metadatas", "distances"]
)

for doc, emb, met, dist in zip(
    results["documents"][0],
    results["embeddings"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(f"  Score : {dist:.4f}")
    print(f"  Doc   : {doc}")
    print(f"  Met   : {met}")
    print(f"  Emb   : {emb[:5]} ...\n")

Query: 'What is machine learning?'
query_embedding:  [-0.01995455101132393, 0.009877976030111313, 0.010249646380543709, 0.029553720727562904, 0.027186432853341103]
  Score : 0.2594
  Doc   : Machine learning uses algorithms to learn from data
  Met   : None
  Emb   : [-0.05912226  0.01819335  0.02581183  0.02106225  0.08048993] ...

  Score : 0.8263
  Doc   : Python is a great programming language
  Met   : None
  Emb   : [-0.04965056 -0.00353516 -0.0051572  -0.00108996 -0.05018304] ...

  Score : 0.9260
  Doc   : ChromaDB stores and queries vector embeddings
  Met   : None
  Emb   : [-0.06050218 -0.03483855 -0.06409699  0.03493492 -0.00735043] ...



### Fetch a document and its embedding

In [19]:
fetched = collection.get(ids=["doc1"], include=["documents", "embeddings"])

print(f"doc1 doc  : {fetched['documents'][0]}")
print(f"doc1 emb  : {fetched['embeddings'][0][:5]}")

doc1 doc  : Python is widely used for AI and data science
doc1 emb  : [-0.08110915 -0.01357943 -0.0153806   0.05772566  0.01544361]


### Update a document with new embedding

In [12]:
new_text = "Python is widely used for AI and data science"
new_embedding = model.encode([new_text]).tolist()

collection.update(
    ids=["doc1"],
    documents=[new_text],
    embeddings=new_embedding,
)

updated = collection.get(ids=["doc1"], include=["documents", "embeddings"])
print(f"Updated text : {updated['documents'][0]}")
print(f"Updated emb  : {updated['embeddings'][0][:5]}")

Updated text : Python is widely used for AI and data science
Updated emb  : [-0.08110915 -0.01357943 -0.0153806   0.05772566  0.01544361]


### Delete a document

In [13]:
collection.delete(ids=["doc4"])
print(f"Deleted doc4. Total remaining: {collection.count()}")

Deleted doc4. Total remaining: 3
